In [ ]:
!pip install -U -q accelerate transformers
!pip install -U -q bitsandbytes
!pip install -q pymupdf
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 85.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import urllib.request
import urllib.parse
from collections import defaultdict, deque
import pandas as pd
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig)
import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize
import fitz
import time
import random

In [ ]:
RELATION_NORMALIZATION = {
    "used": "uses",
    "using": "uses",
    "utilizes": "uses",
    "leverages": "uses",
    "leveraged": "uses",
    "target": "targets",
    "targeted": "targets",
    "targeting": "targets",
    "exploit": "exploits",
    "exploited": "exploits",
    "download": "downloads",
    "downloaded": "downloads",
    "deliver": "delivers",
    "delivered": "delivers",
    "communicates with": "communicates-with",
    "attributed to": "attributed-to",
    "variant of": "variant-of",
    "beacons to": "beacons-to",
    "consists of": "consists-of",
    "exfiltrates to": "exfiltrates-to",
    "originates from": "originates-from",
    "based on": "based-on",
    "duplicate of": "duplicate-of",
    "related to": "related-to",
    "located at": "located-at",
}

ALLOWED_RELATIONS = [
    "uses","exploits", "targets", "attributed-to", "downloads",
    "authored-by", "variant-of", "communicates-with", "delivers",
    "beacons-to", "consists-of", "hosts", "impersonates",
    "exfiltrates-to", "drops", "controls", "compromises",
    "originates-from", "owns", "indicates", "based-on",
    "duplicate-of", "related-to", "located-at",
]

ALLOWED_ENTITIES = sorted([
    "threat-actor", "malware", "tools", "SOFTWARE", "vulnerability",
    "identity", "location", "url", "IPV4", "Infrastucture",
    "attack-pattern", "campaign", "FILEPATH", "REGISTRYKEY",
    "hash", "EMAIL", "TIME"
])

ENTITY_MAP = {e.lower(): e for e in ALLOWED_ENTITIES}

EXTRA_ENTITY_ALIASES = {
    "tool": "tools",
    "threat actor": "threat-actor",
    "attack pattern": "attack-pattern",
    "infra": "Infrastucture",
    "infrastructure": "Infrastucture",
    "file path": "FILEPATH",
    "file-path": "FILEPATH",
    "registry key": "REGISTRYKEY",
    "registry-key": "REGISTRYKEY",
    "ip": "IPV4",
    "ip address": "IPV4",
}

ENTITY_MAP.update(EXTRA_ENTITY_ALIASES)

CTI_KEYWORDS = {
    "malware", "apt", "actor", "exploit", "cve", "attack", "campaign",
    "backdoor", "trojan", "ransomware", "phishing", "c2", "payload",
    "vulnerability", "deploy", "download", "beacon", "exfiltrate",
    "spear", "lateral", "persistence", "credential", "privilege",
}

def is_cti_relevant(sentence: str) -> bool:
    lower = sentence.lower()
    return sum(1 for kw in CTI_KEYWORDS if kw in lower) >= 2

In [ ]:
def get_repo(owner="blackorbird", repo="APT_REPORT",
                  branch="master") -> dict:
    url = (f"https://api.github.com/repos/{owner}/{repo}"
           f"/git/trees/{branch}?recursive=1")
    with urllib.request.urlopen(url) as resp:
        return json.load(resp)

def _norm(s: str) -> str:
    return s.lower().replace(" ", "").replace("-", "").replace("_", "")

def select_files(tree: dict, max_files: int = 100, max_file_size_mb: float = 8) -> list:
    THREAT_ACTORS = {
        "apt28", "apt29", "apt32", "apt33", "apt34", "apt38", "apt41",
        "kimsuky", "lazarus", "sandworm", "gamaredon", "turla",
        "fin7", "fin8", "darkside", "revil", "conti", "muddywater",
    }
    EXCLUDE = {"summary", "cybercrime", "aisecurity", "ot",
               "international strategic"}

    blobs = [
        t for t in tree["tree"]
        if t["type"] == "blob"
        and t["path"].lower().endswith((".pdf", ".txt", ".md"))
    ]

    by_folder: dict = defaultdict(list)
    for f in blobs:
        folder = f["path"].split("/")[0]
        if folder.lower() in EXCLUDE:
            continue
        if f["path"].split("/")[-1].lower() in ("readme.md", "readme.txt"):
            continue
        size_mb = f.get("size", 0) / 1e6
        if size_mb > max_file_size_mb:
            continue
        if _norm(folder) in {_norm(a) for a in THREAT_ACTORS}:
            by_folder[folder].append(f)

    queues = {
        k: deque(sorted(v, key=lambda x: x.get("size", 0)))
        for k, v in by_folder.items()
    }
    order = deque(queues.keys())
    selected = []
    while order and len(selected) < max_files:
        folder = order.popleft()
        q = queues[folder]
        if q:
            selected.append(q.popleft())
            if q:
                order.append(folder)

    groups = sorted({f["path"].split("/")[0] for f in selected})
    print(f"Selected {len(selected)} files across {len(groups)} APT groups:")
    print(" ", groups)
    return selected

def download_files(selected: list,
                   out_dir: str = "apt_subset",
                   owner: str = "blackorbird",
                   repo: str = "APT_REPORT",
                   branch: str = "master") -> list:
    os.makedirs(out_dir, exist_ok=True)
    downloaded = []
    for i, f in enumerate(selected, 1):
        path = f["path"]
        url = (f"https://raw.githubusercontent.com/{owner}/{repo}"
               f"/{branch}/" + urllib.parse.quote(path))
        local = os.path.join(out_dir, path.replace("/", "__"))
        try:
            urllib.request.urlretrieve(url, local)
            downloaded.append(local)
        except Exception as e:
            print(f"  skipped {path}: {e}")
        if i % 10 == 0 or i == len(selected):
            print(f"  downloaded {i}/{len(selected)}")
    return downloaded


print("Fetching file listing from GitHub")
tree = get_repo()
selected = select_files(tree, max_files=100)
report_files = download_files(selected)
print(f"\nReady: {len(report_files)} files downloaded.")

Fetching file listing from GitHub
Selected 96 files across 10 APT groups:
  ['APT28', 'APT29', 'APT34', 'APT41', 'Gamaredon', 'Sandworm', 'Turla', 'kimsuky', 'lazarus', 'muddywater']
  downloaded 10/96
  downloaded 20/96
  downloaded 30/96
  downloaded 40/96
  downloaded 50/96
  downloaded 60/96
  downloaded 70/96
  downloaded 80/96
  downloaded 90/96
  skipped APT28/DoppelgangerNG_ClearSky.pdf: HTTP Error 503: between bytes timeout
  downloaded 96/96

Ready: 95 files downloaded.


In [ ]:
def extract_text(path: str, max_pages: int = 12) -> str:
    if path.lower().endswith(".pdf"):
        try:
            doc = fitz.open(path)
            pages = doc[:max_pages] if len(doc) > max_pages else doc
            return "\n".join(page.get_text() for page in pages)
        except Exception as e:
            print(f"  PDF skipped ({path}): {e}")
            return ""
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        return fh.read()

def build_sentence_pool(files: list, target: int = 5000) -> list:
    pool: set = set()
    for path in files:
        if len(pool) >= target:
            break
        text = extract_text(path)
        if not text.strip():
            continue
        cleaned = re.sub(r"\s+", " ", text)
        for sent in sent_tokenize(cleaned):
            sent = sent.strip()
            if 40 < len(sent) < 600 and is_cti_relevant(sent):
                pool.add(sent)
                if len(pool) >= target:
                    break

    sentences = list(pool)
    print(f"\nSentence pool: {len(sentences)} CTI-relevant sentences "
          f"(from {len(files)} files)")
    return sentences

sentences = build_sentence_pool(report_files, target=5000)


Sentence pool: 1332 CTI-relevant sentences (from 95 files)


In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.7 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
client = Groq(api_key=GROQ_API_KEY)
print("Groq client initialized")

CANDIDATE_MODEL = "qwen/qwen3.6-27b"

Groq client initialized


In [ ]:
import re as _re

# ---------------------------------------------------------------------------
# Cheap, local pre-filter so we only spend Groq calls on sentences that could
# plausibly contain a CTI relation. is_cti_relevant() already narrowed the
# sentence pool by keyword density; this is a second, cheaper pass.
# ---------------------------------------------------------------------------
RULE_BOOK = {
    relation: {"triggers": [relation.replace("-", " ")]}
    for relation in ALLOWED_RELATIONS
}
# fold in the informal/inflected forms from RELATION_NORMALIZATION as extra
# triggers for whichever canonical relation they normalize to
for surface_form, canonical in RELATION_NORMALIZATION.items():
    RULE_BOOK.setdefault(canonical, {"triggers": []})
    RULE_BOOK[canonical]["triggers"].append(surface_form)

REGEX_ENTITY_PATTERNS = {
    "ipv4":        _re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
    "hash":        _re.compile(r"\b[a-fA-F0-9]{32}\b|\b[a-fA-F0-9]{40}\b|\b[a-fA-F0-9]{64}\b"),
    "cve":         _re.compile(r"\bCVE-\d{4}-\d{4,7}\b", _re.IGNORECASE),
    "url":         _re.compile(r"https?://\S+", _re.IGNORECASE),
    "email":       _re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
    "filepath":    _re.compile(r"[A-Za-z]:\\\S+|/[\w./-]+\.\w{1,5}\b"),
    "registrykey": _re.compile(r"\bHK(?:LM|CU|CR|U|CC)\\\S+", _re.IGNORECASE),
}


def should_call_llm(sentence: str) -> bool:
    """Cheap local check: only send a sentence to the LLM if it contains a
    relation-trigger word/phrase OR a regex-detectable entity (IP, hash,
    CVE, URL, email, file path, registry key). Cuts wasted Groq calls on
    sentences that are CTI-flavored (they passed is_cti_relevant) but don't
    actually describe a head-relation-tail fact."""
    text = sentence.lower()

    for relation in RULE_BOOK.values():
        for trigger in relation["triggers"]:
            if trigger.lower() in text:
                return True

    for pattern in REGEX_ENTITY_PATTERNS.values():
        if pattern.search(sentence):
            return True

    return False


In [ ]:
# def build_prompt(sentence: str) -> str:
#     relations_str = ", ".join(ALLOWED_RELATIONS)
#     entities_str = ", ".join(ALLOWED_ENTITIES)

#     content = f"""You are a Cyber Threat Intelligence extraction engine.

# Allowed relation types : {relations_str}
# Allowed entity types   : {entities_str}

# Extract every relationship triple from the sentence below.
# Output ONLY a valid JSON array — no explanation, no markdown:
# [{{"head": "...", "head_type": "...", "relation": "...", "tail": "...", "tail_type": "..."}}]

# If no relation applies, output exactly: []

# NOTE: spell Infrastructure as "Infrastucture" (matches the dataset label).

# Sentence:
# "{sentence}"

# Output:
# """
#     return content

In [ ]:
def build_prompt(sentences: list) -> str:
    relations_str = ", ".join(ALLOWED_RELATIONS)
    entities_str  = ", ".join(ALLOWED_ENTITIES)
    numbered = "\n".join(f'{i+1}. "{s}"' for i, s in enumerate(sentences))

    content = f"""You are a Cyber Threat Intelligence extraction engine.

Allowed relation types : {relations_str}
Allowed entity types   : {entities_str}

You will be given {len(sentences)} numbered sentences.
For EACH sentence extract relation triples. Output ONLY a single valid JSON array
with exactly {len(sentences)} elements (one per sentence, in order).
Each element is an array of triples or [] if none apply.

Triple format: {{"head":"...", "head_type":"...", "relation":"...", "tail":"...", "tail_type":"..."}}

Do NOT output any text outside the JSON array.
NOTE: spell Infrastructure as "Infrastucture".

EXAMPLE for 2 sentences:
[
  [{{"head":"Cobalt Strike","head_type":"tools","relation":"beacons-to","tail":"1.2.3.4","tail_type":"IPV4"}}],
  []
]

Sentences:
{numbered}
"""
    return content


In [ ]:
import threading
import time
import random

# ---------------------------------------------------------------------------
# Rate limiting & Pacing Configuration
# ---------------------------------------------------------------------------
MAX_RETRIES = 6
INITIAL_DELAY = 0.5   # Base delay for exponential backoff (seconds)
MAX_DELAY = 30        # Maximum wait delay (seconds)
MIN_PACING = 4.0      # Minimum gap enforced between requests
DEFAULT_MAX_TOKENS = 600

class RateLimiter:
    """Enforces a minimum pacing between requests and adapts based on 429 errors."""
    def __init__(self, min_pacing: float = MIN_PACING):
        self.min_pacing = min_pacing
        self.current_pacing = min_pacing
        self._last_call = 0.0
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.monotonic()
            remaining = self.current_pacing - (now - self._last_call)
            if remaining > 0:
                time.sleep(remaining)
            self._last_call = time.monotonic()

    def penalize(self):
        with self._lock:
            self.current_pacing = min(self.current_pacing * 1.5, 5.0)

    def relax(self):
        with self._lock:
            self.current_pacing = max(self.min_pacing, self.current_pacing * 0.95)

rate_limiter = RateLimiter()

def call_groq(prompt, max_tokens: int = 600):
    for attempt in range(MAX_RETRIES):
        rate_limiter.wait()
        try:
            response = client.chat.completions.create(
                model=CANDIDATE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=max_tokens,
                reasoning_effort="none",
            )
            rate_limiter.relax()
            return response

        except Exception as e:
            error = str(e)

            # Daily quota exhausted — tell the caller to stop immediately
            if "TPD" in error or "tokens per day" in error.lower():
                print("\n" + "=" * 60)
                print("Groq daily token limit reached. Stopping.")
                print("=" * 60)
                return "DAILY_LIMIT_EXCEEDED"

            retryable = any(code in error for code in ["429", "500", "502", "503", "504"])
            if not retryable:
                raise

            if attempt == MAX_RETRIES - 1:
                print(f"Giving up after {MAX_RETRIES} retries: {error[:200]}")
                return None

            rate_limiter.penalize()
            wait = min(INITIAL_DELAY * (2 ** attempt), MAX_DELAY)
            wait += random.uniform(0, 0.5)
            status = "429" if "429" in error else "server error"
            print(f"{status} received. Retry {attempt+1}/{MAX_RETRIES}, "
                  f"waiting {wait:.1f}s")
            time.sleep(wait)

    return None



In [ ]:
# def _parse_triple(raw_triple: dict, sentence: str) -> dict:
#     # 1. Normalize dictionary keys to lower case
#     clean_dict = {str(k).lower(): v for k, v in raw_triple.items()}

#     # Extract fields flexible to different naming conventions
#     head = clean_dict.get("head") or clean_dict.get("subject") or clean_dict.get("head_entity")
#     head_type = clean_dict.get("head_type") or clean_dict.get("subject_type")
#     relation = clean_dict.get("relation") or clean_dict.get("predicate")
#     tail = clean_dict.get("tail") or clean_dict.get("object") or clean_dict.get("tail_entity")
#     tail_type = clean_dict.get("tail_type") or clean_dict.get("object_type")

#     # 2. Reject missing core entities
#     if not head or not relation or not tail:
#         return None

#     # Clean whitespace
#     head = str(head).strip()
#     relation = str(relation).strip()
#     tail = str(tail).strip()

#     # 3. Optional substring check: don't strictly discard if case doesn't match
#     # Only use this if you want strict grounding in sentence:
#     # if head.lower() not in sentence.lower() or tail.lower() not in sentence.lower():
#     #     return None

#     return {
#         "sentence": sentence,
#         "head": head,
#         "head_type": str(head_type).strip() if head_type else "UNKNOWN",
#         "relation": relation,
#         "tail": tail,
#         "tail_type": str(tail_type).strip() if tail_type else "UNKNOWN",
#     }

In [ ]:
import ast

def extract_first_json_array(text: str):
    """
    Return the substring of the first top-level JSON array found in text
    using a balanced-bracket scan, or None if not found.
    """
    if not isinstance(text, str):
        return None
    start = text.find('[')
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        c = text[i]
        if c == '[':
            depth += 1
        elif c == ']':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def parse_json_array_flexible(text: str):
    """
    Try to parse a JSON array robustly:
      1) extract the first balanced array with extract_first_json_array
      2) try json.loads()
      3) if that fails, try ast.literal_eval after conservative normalization
    Returns the parsed Python object (should be a list) or raises an exception.
    """
    if not isinstance(text, str):
        raise ValueError("Non-string model output")

    arr_txt = extract_first_json_array(text)
    if arr_txt is None:
        # no top-level array found
        raise ValueError("No JSON array found in output")

    try:
        return json.loads(arr_txt)
    except Exception as e_json:
        # as a last-resort fallback try ast.literal_eval after minimal fixes
        # useful when the model emitted single quotes or python-literal dicts
        safe = arr_txt
        # if there are single quotes and no double quotes, try converting single->double
        if "'" in safe and '"' not in safe:
            safe = safe.replace("'", '"')
        # remove common trailing commas inside arrays/objects (simple heuristic)
        safe = safe.replace(",]", "]").replace(",}", "}")
        try:
            parsed = ast.literal_eval(safe)
            return parsed
        except Exception as e_ast:
            # attach both exception messages for debugging
            raise RuntimeError(f"json.loads error: {e_json}; ast.literal_eval error: {e_ast}")



In [ ]:
def strip_thinking_block(text: str) -> str:
    """Qwen3 and other reasoning models wrap their output in <think>...</think>
    before the actual answer. Strip everything up to and including </think>
    so JSON extraction only looks at the real output."""
    if not isinstance(text, str):
        return text
    think_end = text.find("</think>")
    if think_end != -1:
        return text[think_end + len("</think>"):].strip()
    return text

In [ ]:
def generate_candidates(
    sentences: list,
    batch_size: int = 8,
    max_tokens_per_sentence: int = 300,
    output_path: str = "raw_llm_candidates.json",
) -> pd.DataFrame:
    """Processes sentences in batches of batch_size per API call.
    Stops the moment the daily token limit is hit, saves whatever
    was collected, and returns it so verification can start immediately."""

    rows = []
    idx = 0
    total = len(sentences)

    print(f"Starting batch generation: {total} sentences, "
          f"batch_size={batch_size}, "
          f"max_tokens_per_call={batch_size * max_tokens_per_sentence}")

    while idx < total:
        batch = sentences[idx: idx + batch_size]
        prompt = build_prompt(batch)
        max_tokens = max_tokens_per_sentence * len(batch)

        response = call_groq(prompt, max_tokens=max_tokens)

        # Daily limit hit — save everything collected so far and stop
        if response == "DAILY_LIMIT_EXCEEDED":
            print(f"\nStopped at sentence {idx}/{total}.")
            print(f"Saving {len(rows)} collected triples to {output_path}...")
            pd.DataFrame(rows).to_json(output_path, orient="records", indent=2)
            print("Saved. Proceed to the verification cell.")
            return pd.DataFrame(rows)

        if response is None:
            print(f"No response for batch at sentence {idx}, skipping.")
            idx += len(batch)
            continue

        try:
            gen_text = (response.choices[0].message.content or "").strip()
        except Exception:
            idx += len(batch)
            continue

        if not gen_text:
            idx += len(batch)
            continue

        start_json = gen_text.find("[")
        end_json   = gen_text.rfind("]")

        if start_json == -1 or end_json == -1:
            print(f"No JSON array at sentence {idx}. "
                  f"Snippet: {gen_text[:200]}")
            idx += len(batch)
            continue

        try:
            batch_results = json.loads(gen_text[start_json: end_json + 1])
        except Exception as e:
            print(f"JSON parse error at sentence {idx}: {e}")
            idx += len(batch)
            continue

        if not isinstance(batch_results, list):
            idx += len(batch)
            continue

        if len(batch_results) != len(batch):
            print(f"Warning: model returned {len(batch_results)} elements "
                  f"for {len(batch)} sentences at idx {idx}")

        for sentence, triples in zip(batch, batch_results):
            if not isinstance(triples, list):
                continue
            for triple in triples:
                if not isinstance(triple, dict):
                    continue
                required = ["head", "head_type", "relation", "tail", "tail_type"]
                if not all(k in triple for k in required):
                    continue

                relation  = str(triple["relation"]).lower().strip()
                relation  = RELATION_NORMALIZATION.get(relation, relation)
                if relation not in ALLOWED_RELATIONS:
                    continue

                head_type = str(triple["head_type"]).lower().strip()
                tail_type = str(triple["tail_type"]).lower().strip()
                if head_type not in ENTITY_MAP or tail_type not in ENTITY_MAP:
                    continue

                triple["head_type"] = ENTITY_MAP[head_type]
                triple["tail_type"] = ENTITY_MAP[tail_type]
                triple["relation"]  = relation
                triple["head"]      = str(triple["head"]).strip()
                triple["tail"]      = str(triple["tail"]).strip()

                if not triple["head"] or not triple["tail"]:
                    continue

                triple["sentence"] = sentence
                rows.append(triple)

        idx += len(batch)
        print(f"Processed {idx}/{total} | "
              f"pacing={rate_limiter.current_pacing:.2f}s | "
              f"triples so far={len(rows)}")

    # Completed normally (all sentences processed)
    print(f"\nGeneration complete. {len(rows)} triples extracted.")
    pd.DataFrame(rows).to_json(output_path, orient="records", indent=2)
    print(f"Saved to {output_path}. Proceed to the verification cell.")
    return pd.DataFrame(rows)


In [ ]:
print(f"Running generation on {len(sentences)} sentences (model={CANDIDATE_MODEL})\n")

start_time = time.time()

# Run candidate extraction
raw_candidates_df = generate_candidates(
    sentences,
    batch_size=8,
    max_tokens_per_sentence=300,
)

elapsed = time.time() - start_time
print(f"\nGeneration stopped after {elapsed:.1f} seconds.")
print(f"Total candidate triples extracted: {len(raw_candidates_df)}")

# Save directly to final JSON file for verification
output_filename = "raw_llm_candidates.json"
raw_candidates_df.to_json(output_filename, orient="records", indent=2)

Running generation on 1332 sentences (model=qwen/qwen3.6-27b)

Starting batch generation: 1332 sentences, batch_size=8, max_tokens_per_call=2400
Processed 8/1332 | pacing=4.00s | triples so far=12
Processed 16/1332 | pacing=4.00s | triples so far=29
Processed 24/1332 | pacing=4.00s | triples so far=42
Processed 32/1332 | pacing=4.00s | triples so far=43
Processed 40/1332 | pacing=4.00s | triples so far=59
Processed 48/1332 | pacing=4.00s | triples so far=72
Processed 56/1332 | pacing=4.00s | triples so far=82
Processed 64/1332 | pacing=4.00s | triples so far=89
Processed 72/1332 | pacing=4.00s | triples so far=99
Processed 80/1332 | pacing=4.00s | triples so far=112
Processed 88/1332 | pacing=4.00s | triples so far=119
Processed 96/1332 | pacing=4.00s | triples so far=134
Processed 104/1332 | pacing=4.00s | triples so far=153
Processed 112/1332 | pacing=4.00s | triples so far=161
Processed 120/1332 | pacing=4.00s | triples so far=178
Processed 128/1332 | pacing=4.00s | triples so far=1

In [ ]:
print("Sample candidates")
if not raw_candidates_df.empty:
    cols = ["head", "head_type", "relation", "tail", "tail_type"]
    print(raw_candidates_df[cols].head(20).to_string(index=False))
    print("\nRelation distribution:")
    print(raw_candidates_df["relation"].value_counts().to_string())
    print("\nHead type distribution:")
    print(raw_candidates_df["head_type"].value_counts().to_string())
else:
    print("No candidates extracted.")

raw_candidates_df.to_json("raw_llm_candidates.json", orient="records", indent=2)
print("\nSaved raw_llm_candidates.json")

Sample candidates
                  head     head_type       relation                                                                                      tail      tail_type
                    C2 Infrastucture        targets                                                                              mail servers       identity
                    C2 Infrastucture       exploits                                                                            CVE-2021-26855  vulnerability
                    C2 Infrastucture       exploits                                                                            CVE-2021-26857  vulnerability
                    C2 Infrastucture       exploits                                                                            CVE-2021-26858  vulnerability
                    C2 Infrastucture       exploits                                                                            CVE-2021-27065  vulnerability
         Cyclops Blink       malware    

In [ ]:
# MITRE ATTACK INCLUDED

In [ ]:
# ---------------------------------------------------------------------------
# MITRE ATT&CK ground-truth reference (STIX 2.1 data)
# https://github.com/mitre-attack/attack-stix-data
# ---------------------------------------------------------------------------
ATTACK_STIX_URLS = {
    "enterprise": "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json",
    "mobile":     "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/mobile-attack/mobile-attack.json",
    "ics":        "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/ics-attack/ics-attack.json",
}


def load_attack_stix(domains=("enterprise",), cache_dir="attack_stix"):
    """Downloads (and locally caches) MITRE ATT&CK STIX bundles and returns
    the flat list of STIX objects across all requested domains."""
    os.makedirs(cache_dir, exist_ok=True)
    objects = []
    for domain in domains:
        url = ATTACK_STIX_URLS[domain]
        local_path = os.path.join(cache_dir, f"{domain}-attack.json")
        if not os.path.exists(local_path):
            print(f"Downloading {domain} ATT&CK STIX bundle...")
            urllib.request.urlretrieve(url, local_path)
        with open(local_path, "r", encoding="utf-8") as fh:
            bundle = json.load(fh)
        objects.extend(bundle["objects"])
        print(f"  {domain}: {len(bundle['objects'])} STIX objects loaded")
    return objects


# Map our extraction entity types -> the STIX object type we validate against
ATTACK_TYPE_MAP = {
    "attack-pattern": "attack-pattern",   # techniques / sub-techniques
    "malware":        "malware",
    "tools":          "tool",
    "threat-actor":   "intrusion-set",
    "campaign":       "campaign",
}


def build_attack_kb(stix_objects: list) -> dict:
    """Builds {stix_type: {normalized_name: (canonical_name, external_id)}},
    including all name aliases, for fast lookup/verification of extracted
    entities against the official ATT&CK dataset."""
    kb = defaultdict(dict)
    for obj in stix_objects:
        obj_type = obj.get("type")
        if obj_type not in ATTACK_TYPE_MAP.values():
            continue
        if obj.get("revoked") or obj.get("x_mitre_deprecated"):
            continue

        ext_id = next(
            (r["external_id"] for r in obj.get("external_references", [])
             if r.get("source_name") == "mitre-attack"),
            None,
        )
        name = obj.get("name", "")
        names = {name, *obj.get("aliases", []), *obj.get("x_mitre_aliases", [])}
        for n in names:
            if n:
                kb[obj_type][_norm(n)] = (name, ext_id)
    return kb


print("Loading MITRE ATT&CK STIX reference data for verification...")
attack_stix_objects = load_attack_stix(domains=("enterprise",))
attack_kb = build_attack_kb(attack_stix_objects)
for _t, _d in attack_kb.items():
    print(f"  {_t}: {len(_d)} known names/aliases")


Loading MITRE ATT&CK STIX reference data for verification...
  enterprise: 25920 STIX objects loaded
  campaign: 60 known names/aliases
  intrusion-set: 598 known names/aliases
  malware: 987 known names/aliases
  tool: 114 known names/aliases
  attack-pattern: 672 known names/aliases


In [ ]:
import difflib

def attack_kb_lookup(name: str, entity_type: str, cutoff: float = 0.85):
    """Checks whether `name` matches a known ATT&CK object of the given
    (our) entity_type. Returns (matched, canonical_name, external_id, score)."""
    stix_type = ATTACK_TYPE_MAP.get(entity_type)
    if stix_type is None or stix_type not in attack_kb:
        return False, None, None, 0.0

    table = attack_kb[stix_type]
    norm_name = _norm(name)

    # exact match (case/whitespace/punctuation-insensitive) first
    if norm_name in table:
        canonical, ext_id = table[norm_name]
        return True, canonical, ext_id, 1.0

    # fuzzy fallback, e.g. "Cobalt Strike Beacon" vs "Cobalt Strike"
    match = difflib.get_close_matches(norm_name, table.keys(), n=1, cutoff=cutoff)
    if match:
        canonical, ext_id = table[match[0]]
        score = difflib.SequenceMatcher(None, norm_name, match[0]).ratio()
        return True, canonical, ext_id, round(score, 3)

    return False, None, None, 0.0


def annotate_with_attack_kb(df: pd.DataFrame) -> pd.DataFrame:
    """Adds ATT&CK ground-truth match columns for the head and tail entities
    of every candidate triple (only meaningful for entity types ATT&CK
    covers: attack-pattern, malware, tools, threat-actor, campaign)."""
    if df.empty:
        return df

    df = df.copy()
    head_matches = df.apply(lambda r: attack_kb_lookup(r["head"], r["head_type"]), axis=1)
    tail_matches = df.apply(lambda r: attack_kb_lookup(r["tail"], r["tail_type"]), axis=1)

    df["head_attack_match"] = [m[0] for m in head_matches]
    df["head_attack_name"] = [m[1] for m in head_matches]
    df["head_attack_id"] = [m[2] for m in head_matches]
    df["tail_attack_match"] = [m[0] for m in tail_matches]
    df["tail_attack_name"] = [m[1] for m in tail_matches]
    df["tail_attack_id"] = [m[2] for m in tail_matches]
    return df


raw_candidates_df = annotate_with_attack_kb(raw_candidates_df)

n_grounded = int(raw_candidates_df["head_attack_match"].sum() + raw_candidates_df["tail_attack_match"].sum()) if not raw_candidates_df.empty else 0
print(f"ATT&CK-grounded entity mentions found in candidates: {n_grounded}")


ATT&CK-grounded entity mentions found in candidates: 465


In [ ]:
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")  # store your HF token in Colab secrets
login(token=HF_TOKEN)

In [ ]:
VERIFIER_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {VERIFIER_MODEL_ID} in 4-bit")
verifier_tokenizer = AutoTokenizer.from_pretrained(VERIFIER_MODEL_ID, use_fast=True)
verifier_tokenizer.padding_side = "left"
if verifier_tokenizer.pad_token is None:
    verifier_tokenizer.pad_token = verifier_tokenizer.eos_token

verifier_model = AutoModelForCausalLM.from_pretrained(
    VERIFIER_MODEL_ID,
    quantization_config=quant_cfg,
    device_map="auto",
)
verifier_model.eval()
print("Verifier model loaded.")

Loading meta-llama/Llama-3.1-8B-Instruct in 4-bit


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Verifier model loaded.


In [ ]:
def build_verification_prompt(
    head: str,
    head_type: str,
    relation: str,
    tail: str,
    tail_type: str,
    sentence: str,
) -> list:
    content = f"""
You are a Cyber Threat Intelligence (CTI) relation verification expert.

Your task is to verify whether an extracted relation triple is correct.

Sentence:
"{sentence}"

Extracted Triple:

Head Entity:
"{head}"

Head Entity Type:
"{head_type}"

Relation:
"{relation}"

Tail Entity:
"{tail}"

Tail Entity Type:
"{tail_type}"

Verify whether this triple is factually and semantically supported by the sentence.

Respond ONLY with a valid JSON object in this exact format, no explanation, no markdown:
{{"is_valid": true or false, "confidence": a number between 0 and 1, "reason": "short justification"}}
"""
    return [{"role": "user", "content": content}]

In [ ]:
def verify_all_candidates_local(
    df: pd.DataFrame,
    batch_size: int = 8,
    max_new_tokens: int = 200,
) -> pd.DataFrame:
    """Runs local batched verification across all candidate triples on the T4.
    For each triple, asks Llama 3.1 whether the triple is supported by its
    source sentence. Returns the input df with three new columns added:
      - verifier_is_valid   (bool)
      - verifier_confidence (float 0-1)
      - verifier_reason     (string)
    """
    if df.empty:
        print("No candidates to verify.")
        return df

    records = df.to_dict(orient="records")
    results = []

    print(f"Verifying {len(records)} candidates "
          f"(batch_size={batch_size}, model={VERIFIER_MODEL_ID})...")
    start_time = time.time()

    for start in range(0, len(records), batch_size):
        batch = records[start : start + batch_size]

        # Build one prompt per triple using the chat template
        prompts = [
            verifier_tokenizer.apply_chat_template(
                build_verification_prompt(
                    head=r.get("head", ""),
                    head_type=r.get("head_type", ""),
                    relation=r.get("relation", ""),
                    tail=r.get("tail", ""),
                    tail_type=r.get("tail_type", ""),
                    sentence=r.get("sentence", ""),
                ),
                tokenize=False,
                add_generation_prompt=True,
            )
            for r in batch
        ]

        try:
            inputs = verifier_tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048,
            ).to(verifier_model.device)

            with torch.no_grad():
                outputs = verifier_model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    temperature=None,
                    top_p=None,
                    pad_token_id=verifier_tokenizer.pad_token_id,
                )

            # Decode only the newly generated tokens (not the prompt)
            gen_only = outputs[:, inputs["input_ids"].shape[1]:]
            gen_texts = verifier_tokenizer.batch_decode(
                gen_only, skip_special_tokens=True
            )

        except Exception as e:
            print(f"Batch error at index {start}: {e}")
            for r in batch:
                r["verifier_is_valid"]   = False
                r["verifier_confidence"] = 0.0
                r["verifier_reason"]     = f"Batch error: {str(e)}"
                results.append(r)
            continue

        for r, gen_text in zip(batch, gen_texts):
            gen_text = gen_text.strip()
            start_json = gen_text.find("{")
            end_json   = gen_text.rfind("}")
            try:
                if start_json == -1 or end_json == -1:
                    raise ValueError("No JSON object found in output")
                result = json.loads(gen_text[start_json : end_json + 1])
                r["verifier_is_valid"]   = result.get("is_valid", False)
                r["verifier_confidence"] = result.get("confidence", 0.0)
                r["verifier_reason"]     = result.get("reason", "")
            except Exception as e:
                r["verifier_is_valid"]   = False
                r["verifier_confidence"] = 0.0
                r["verifier_reason"]     = f"Parse error: {str(e)}"
            results.append(r)

        done = min(start + batch_size, len(records))
        print(f"  Verified {done}/{len(records)}")

    elapsed = time.time() - start_time
    print(f"Verification done in {elapsed:.1f}s")
    return pd.DataFrame(results)


In [ ]:
# Update Cell 24 (Verification processing)
verified_df = verify_all_candidates_local(
    raw_candidates_df, batch_size=8, max_new_tokens=200
)

# Ensure required columns exist even if DataFrame is empty
for col in ["verifier_is_valid", "verifier_confidence", "head_attack_match", "tail_attack_match"]:
    if col not in verified_df.columns:
        verified_df[col] = False if "match" in col or "valid" in col else 0.0

is_attack_grounded = (
    verified_df["head_attack_match"].astype(bool) | verified_df["tail_attack_match"].astype(bool)
)

valid_pseudo_labels = verified_df[
    (verified_df["verifier_is_valid"] == True)
    & (
        (verified_df["verifier_confidence"] >= 0.90)
        | ((verified_df["verifier_confidence"] >= 0.75) & is_attack_grounded)
    )
].copy()

print(f"\n--- Verification Summary ---")
print(f"Total Candidates Input        : {len(verified_df)}")
print(f"ATT&CK-Grounded Candidates    : {int(is_attack_grounded.sum())}")
print(f"Validated Pseudo-Labels       : {len(valid_pseudo_labels)}")
print(f"Filtered Out (Noise)          : {len(verified_df) - len(valid_pseudo_labels)}")

Verifying 1274 candidates (batch_size=8, model=meta-llama/Llama-3.1-8B-Instruct)...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  Verified 8/1274
  Verified 16/1274
  Verified 24/1274
  Verified 32/1274
  Verified 40/1274
  Verified 48/1274
  Verified 56/1274
  Verified 64/1274
  Verified 72/1274
  Verified 80/1274
  Verified 88/1274
  Verified 96/1274
  Verified 104/1274
  Verified 112/1274
  Verified 120/1274
  Verified 128/1274
  Verified 136/1274
  Verified 144/1274
  Verified 152/1274
  Verified 160/1274
  Verified 168/1274
  Verified 176/1274
  Verified 184/1274
  Verified 192/1274
  Verified 200/1274
  Verified 208/1274
  Verified 216/1274
  Verified 224/1274
  Verified 232/1274
  Verified 240/1274
  Verified 248/1274
  Verified 256/1274
  Verified 264/1274
  Verified 272/1274
  Verified 280/1274
  Verified 288/1274
  Verified 296/1274
  Verified 304/1274
  Verified 312/1274
  Verified 320/1274
  Verified 328/1274
  Verified 336/1274
  Verified 344/1274
  Verified 352/1274
  Verified 360/1274
  Verified 368/1274
  Verified 376/1274
  Verified 384/1274
  Verified 392/1274
  Verified 400/1274
  Verified 40

In [ ]:
# verified_df = verify_all_candidates_local(
#     raw_candidates_df, batch_size=8, max_new_tokens=200
# )

# # An extraction is accepted if either:
# #  (a) the local LLM verifier is confident (>= 0.90), OR
# #  (b) the LLM verifier agrees it's valid AND at least one of the entities is
# #      grounded in the official MITRE ATT&CK STIX data (>= 0.75 confidence is
# #      enough in that case, since the KB match is independent corroboration)
# is_attack_grounded = (
#     verified_df.get("head_attack_match", False) | verified_df.get("tail_attack_match", False)
# )

# valid_pseudo_labels = verified_df[
#     (verified_df["verifier_is_valid"] == True)
#     & (
#         (verified_df["verifier_confidence"] >= 0.90)
#         | ((verified_df["verifier_confidence"] >= 0.75) & is_attack_grounded)
#     )
# ].copy()

# print(f"\n--- Verification Summary ---")
# print(f"Total Candidates Input        : {len(verified_df)}")
# print(f"ATT&CK-Grounded Candidates    : {int(is_attack_grounded.sum())}")
# print(f"Validated Pseudo-Labels       : {len(valid_pseudo_labels)}")
# print(f"Filtered Out (Noise)          : {len(verified_df) - len(valid_pseudo_labels)}")


In [ ]:
# Update Cell 25 (Statistics reporting)
print("\n========== Statistics ==========")

print(f"Total candidates: {len(raw_candidates_df)}")

if not raw_candidates_df.empty and "relation" in raw_candidates_df.columns:
    print(f"Unique relations: {raw_candidates_df['relation'].nunique()}")
    print("\nAccepted relation distribution:")
    print(valid_pseudo_labels["relation"].value_counts() if not valid_pseudo_labels.empty else "None")
    print("\nRejected relation distribution:")
    print(
        verified_df[verified_df["verifier_is_valid"] == False]["relation"].value_counts()
        if not verified_df.empty else "None"
    )
else:
    print("No valid extracted triples to show statistics for.")

verified_df.to_json("groq_verified_full_log.json", orient="records", indent=2)
valid_pseudo_labels.to_json("verified_pseudo_labels.json", orient="records", indent=2)
print("\nSaved 'groq_verified_full_log.json' and 'verified_pseudo_labels.json'")


========== Statistics ==========
Total candidates: 1274
Unique relations: 24

Accepted relation distribution:
relation
uses                 256
targets              177
exploits              77
related-to            44
attributed-to         39
downloads             38
drops                 32
delivers              27
indicates             25
authored-by           21
exfiltrates-to        18
communicates-with     18
duplicate-of          17
hosts                 11
beacons-to            10
compromises            9
impersonates           8
consists-of            6
controls               4
located-at             4
originates-from        4
owns                   3
based-on               2
variant-of             1
Name: count, dtype: int64

Rejected relation distribution:
relation
uses                 73
targets              34
communicates-with    31
downloads            31
drops                26
indicates            26
hosts                23
located-at           21
exploits            

In [ ]:
## Converting LLM extractions to match the labelled dataset format

In [ ]:
# Same marker-insertion logic used in Baseline_Training.ipynb, copied here so the
# LLM-extracted triples end up marked up identically to the training data.
def insert_entity_markers(text, head_start, head_end, tail_start, tail_end):
    spans = sorted([(head_start, head_end, "[E1]", "[/E1]"),
                    (tail_start, tail_end, "[E2]", "[/E2]")],
                   key=lambda s: -s[0])  # right-to-left so inserted tags don't shift earlier offsets
    out = text
    for start, end, open_tag, close_tag in spans:
        out = out[:end] + f" {close_tag}" + out[end:]
        out = out[:start] + f"{open_tag} " + out[start:]
    return out


In [ ]:
import re as _re2

def find_entity_span(sentence: str, entity: str):
    """Locate `entity` inside `sentence` and return (start, end) character
    offsets, or None if it can't be found. Tries, in order:
      1) exact case-insensitive substring match
      2) match after collapsing internal whitespace (LLM output sometimes
         normalizes spacing differently from the source sentence)
    Only the first occurrence is used.
    """
    if not entity or not sentence:
        return None

    entity = entity.strip()
    if not entity:
        return None

    # 1) exact case-insensitive match
    match = _re2.search(_re2.escape(entity), sentence, flags=_re2.IGNORECASE)
    if match:
        return match.start(), match.end()

    # 2) collapse whitespace differences and retry
    loose_pattern = _re2.escape(entity)
    loose_pattern = _re2.sub(r"\\ ", r"\\s+", loose_pattern)
    match = _re2.search(loose_pattern, sentence, flags=_re2.IGNORECASE)
    if match:
        return match.start(), match.end()

    return None


def spans_overlap(a_start, a_end, b_start, b_end) -> bool:
    return a_start < b_end and b_start < a_end


In [ ]:
def convert_llm_output_to_labelled_format(df: pd.DataFrame) -> pd.DataFrame:
    """Reshapes LLM-extracted triples (head/head_type/relation/tail/tail_type/sentence)
    into the same relation-level schema as `all_df` in Baseline_Training.ipynb
    (marked_text/head_type/tail_type/relation), by locating the head and tail
    entity mentions in the source sentence and inserting [E1]/[E2] markers.

    Rows are dropped (and counted) when:
      - the head or tail text can't be located in the sentence, or
      - the head and tail spans overlap (marker insertion would be ambiguous)
    """
    if df.empty:
        return pd.DataFrame(columns=["marked_text", "head_type", "tail_type", "relation"])

    rows = []
    dropped_not_found = 0
    dropped_overlap = 0

    for _, r in df.iterrows():
        sentence = r.get("sentence", "")
        head = r.get("head", "")
        tail = r.get("tail", "")

        head_span = find_entity_span(sentence, head)
        tail_span = find_entity_span(sentence, tail)

        if head_span is None or tail_span is None:
            dropped_not_found += 1
            continue

        head_start, head_end = head_span
        tail_start, tail_end = tail_span

        if spans_overlap(head_start, head_end, tail_start, tail_end):
            dropped_overlap += 1
            continue

        marked_text = insert_entity_markers(sentence, head_start, head_end, tail_start, tail_end)

        rows.append({
            "marked_text": marked_text,
            "head_type": r.get("head_type", ""),
            "tail_type": r.get("tail_type", ""),
            "relation": r.get("relation", ""),
        })

    print(f"Converted {len(rows)}/{len(df)} triples to labelled format.")
    print(f"  Dropped (entity text not found in sentence): {dropped_not_found}")
    print(f"  Dropped (head/tail spans overlap): {dropped_overlap}")

    return pd.DataFrame(rows, columns=["marked_text", "head_type", "tail_type", "relation"])


In [ ]:
# Use the verified, high-confidence pseudo-labels as the source (falls back to
# all raw candidates if verification hasn't been run / produced nothing).
source_df = valid_pseudo_labels if "valid_pseudo_labels" in globals() and not valid_pseudo_labels.empty else raw_candidates_df

llm_labelled_df = convert_llm_output_to_labelled_format(source_df)

print("\nRelation distribution in converted set:")
print(llm_labelled_df["relation"].value_counts().to_string())

llm_labelled_df.to_json("llm_extracted_labelled_format.json", orient="records", indent=2)
print("\nSaved llm_extracted_labelled_format.json — same schema as `all_df` in Baseline_Training.ipynb")
print("(marked_text, head_type, tail_type, relation). Ready to be label-encoded with the")
print("same LabelEncoder and passed through the trained tokenizer/model for validation.")


Converted 826/851 triples to labelled format.
  Dropped (entity text not found in sentence): 23
  Dropped (head/tail spans overlap): 2

Relation distribution in converted set:
relation
uses                 248
targets              174
exploits              70
related-to            44
attributed-to         38
downloads             37
drops                 31
delivers              25
indicates             25
authored-by           21
exfiltrates-to        18
communicates-with     18
duplicate-of          17
beacons-to            10
hosts                 10
compromises            9
impersonates           8
consists-of            6
located-at             4
controls               4
originates-from        4
owns                   3
based-on               2

Saved llm_extracted_labelled_format.json — same schema as `all_df` in Baseline_Training.ipynb
(marked_text, head_type, tail_type, relation). Ready to be label-encoded with the
same LabelEncoder and passed through the trained tokenizer/mode

In [ ]:
from google.colab import files
# files.download("raw_llm_candidates .json")
files.download("verified_pseudo_labels.json")
files.download("llm_extracted_labelled_format.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>